# Lecture 3: tests

This lecture is based onto [https://realpython.com/python-testing/](https://realpython.com/python-testing/). 
Honestly, *realpython* is a very valuable source of documentation about Python, and reading their pages is always a pleasure.

As its title indicates it, this lecture is about testing in Python. Testing what? No, covid is old story, we want here to test Python code. &#x1F609;

## Objectives
By the end of this lecture you can:
- distinguish **manual vs automated** testing and **unit vs integration** tests;
- write tests with **`unittest`** (a `TestCase` subclass, test methods, `unittest.main()`);
- pick the right **assertion** (`assertEqual`, `assertTrue`, `assertRaises`, `assertIsInstance`, ...);
- run tests three ways -- as a script, with `python -m unittest`, and by **discovery** (`-s tests`) -- and read the output;
- handle **expected exceptions** with `assertRaises`, and understand **fixtures** and **parameterization**;
- spot **side effects** (and their link to the Single Responsibility Principle) and isolate them with **`unittest.mock`** (`Mock`, `MagicMock`, `patch`, `side_effect`);
- write the same tests with **`pytest`** -- plain functions, bare `assert`, `pytest.raises`, `pytest.approx`, `@pytest.fixture` and `conftest.py`, `@pytest.mark.parametrize` and markers -- and know that `pytest` runs your `unittest` tests unchanged.

## Testing Your Code
There are many ways to test your code. 

### Automated vs. Manual Testing
The good news is, you’ve probably already created a test without realizing it. Remember when you ran your application and used it for the first time? Did you check the features and experiment using them? That’s known as **exploratory testing** and is a form of manual testing.

Exploratory testing is a form of testing that is done without a plan. In an exploratory test, you’re just exploring the application...

To have a complete set of manual tests, all you need to do is make a list of all the features your application has, the different types of input it can accept, and the expected results. Now, every time you make a change to your code, you need to go through every single item on that list and check it like any airplane pilot does before to take off.

That doesn’t sound like much fun, does it?

This is where **automated testing** comes in. Automated testing is the execution of your test plan (the parts of your application you want to test, the order in which you want to test them, and the expected responses) by a script instead of a human. Python already comes with a set of tools and libraries to help you create automated tests for your application. 

### Unit Tests vs. Integration Tests

The world of testing has no shortage of terminology, and now that you know the difference between automated and manual testing, it’s time to go a level deeper.

Think of how you might test the lights on a car. You would turn on the lights (known as the test step) and go outside the car or ask a friend to check that the lights are on (known as the test assertion). **Testing multiple components at once is known as integration testing**.

Think of all the things that need to work correctly in order for a simple task to give the right result. These components are like the parts to your application, all of those classes, functions, and modules you’ve written.

*A major challenge with integration testing is when an integration test doesn’t give the right result*. It’s very hard to diagnose the issue without being able to isolate which part of the system is failing. If the lights didn’t turn on, then maybe the bulbs are broken. Is the battery dead? What about the alternator? Is the car’s computer failing?

If you have a fancy modern car, it will tell you when your light bulbs have gone. It does this using a form of unit test. &#x1F60F;

**A unit test is a smaller test**, one that checks that a single component operates in the right way. A unit test helps you to isolate what is broken in your application and fix it faster.

You have just seen two types of tests:
1. An integration test checks that components in your application operate with each other.
2. A unit test checks a small component in your application.

You can write both integration tests and unit tests in Python. To write a unit test for the built-in function `sum()`, you would check the output of `sum()` against a known output.

For example, here’s how you check that the `sum()` of the numbers `(1, 2, 3)` equals `6`:
```python
>>> assert sum([1, 2, 3]) == 6, "Should be 6"
```
This will not output anything on the REPL because the values are correct.

If the result from `sum()` is incorrect, this will fail with an `AssertionError` and the message `"Should be 6"`. Try an assertion statement again with the wrong values to see an AssertionError:
```python
>>> assert sum([1, 1, 1]) == 6, "Should be 6"
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
AssertionError: Should be 6
```
In the REPL, you are seeing the raised `AssertionError` because the result of `sum()` does not match `6`.

Instead of testing on the REPL, you’ll want to put this into a new Python file called `test_sum.py` and execute it again:
```python
def test_sum():
    assert sum([1, 2, 3]) == 6, "Should be 6"

if __name__ == "__main__":
    test_sum()
    print("Everything passed")
```
Now you have written a test case, an assertion, and an entry point (the command line). You can now execute this at the command line:
```bash
$ python test_sum.py
Everything passed
```
You can see the successful result, `Everything passed`.

In Python, `sum()` accepts any iterable as its first argument. You tested with a list. Now test with a tuple as well. Create a new file called `test_sum_2.py` with the following code:
```python
def test_sum():
    assert sum([1, 2, 3]) == 6, "Should be 6"

def test_sum_tuple():
    assert sum((1, 2, 2)) == 6, "Should be 6"

if __name__ == "__main__":
    test_sum()
    test_sum_tuple()
    print("Everything passed")
```
When you execute `test_sum_2.py`, the script will give an error because the `sum()` of `(1, 2, 2)` is `5`, not `6`. The result of the script gives you the error message, the line of code, and the traceback:
```bash
$ python test_sum_2.py
Traceback (most recent call last):
  File "test_sum_2.py", line 9, in <module>
    test_sum_tuple()
  File "test_sum_2.py", line 5, in test_sum_tuple
    assert sum((1, 2, 2)) == 6, "Should be 6"
AssertionError: Should be 6
```
Here you can see how a mistake in your code gives an error on the console with some information on where the error was and what the expected result was.

Writing tests in this way is okay for a simple check, but what if more than one fails? This is where **test runners** come in. The test runner is a special application designed for running tests, checking the output, and giving you tools for debugging and diagnosing tests and applications.

There are different test runners in Python. In this lecture we are used the basic one: `unittest`.


##  Testing with `unittest`
`unittest` has been built into the Python standard library since version 2.1. 
You will probably see it in commercial Python applications and open-source projects.

`unittest` contains both a testing framework and a test runner. 
It has some important requirements for writing and executing tests.

It requires that:
- You put your tests into classes as methods.
- You use a series of special assertion methods in the `unittest.TestCase` class instead of the built-in `assert` statement.

To convert the earlier example to a `unittest` test case, you would have to:
- Import `unittest` from the standard library.
- Create a class called `TestSum` that inherits from the `TestCase` class.
- Convert the `test` functions into methods by adding `self` as the first argument.
- Change the assertions to use the `self.assertEqual()` method on the `TestCase` class.
- Change the command-line entry point to call `unittest.main()`.

Follow those steps by creating a new file `test_sum_unittest.py` with the code provided in the next cell.

In [ ]:
import os, sys

# The interpreter that is running this notebook.
# On macOS there is often no `python` command at all, only `python3`, and adding a
# shell alias does not help: `!` cells run a NON-interactive shell that never reads
# your profile, and `%%script` launches the program directly, without any shell.
# `sys.executable` is an absolute path, so it keeps working after a `cd` too.
PY = sys.executable

# Everything this notebook writes goes into code/, which is git-ignored.
# It is scratch: re-run the notebook to regenerate it, and never commit it.
os.makedirs('code', exist_ok=True)
print(f'interpreter: {PY}')
print(f'writing into: {os.path.abspath("code")}')


In [5]:
%%writefile code/test_sum_unittest.py
import unittest

class TestSum(unittest.TestCase):
    def test_sum(self):
        self.assertEqual(sum([1, 2, 3]), 6, "Should be 6")

    def test_sum_tuple(self):
        # intentionally wrong: (1, 2, 2) sums to 5, so this test FAILS on purpose
        self.assertEqual(sum((1, 2, 2)), 6, "Should be 6")

if __name__ == '__main__':
    unittest.main()

Writing test_sum_unittest.py


When you execute this in Python (use the next cell for this), you see one success (indicated with `.`) and one failure (indicated with `F`)...

In [7]:
!cd code && {PY} test_sum_unittest.py

.F
FAIL: test_sum_tuple (__main__.TestSum)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "d:\laveneau\OneDrive\OneDrive - Université de Poitiers\Enseignement\USTH\ProgrammingTechniques\Lectures\Lecture3\test_sum_unittest.py", line 8, in test_sum_tuple
    self.assertEqual(sum((1, 2, 2)), 6, "Should be 6")
AssertionError: 5 != 6 : Should be 6

----------------------------------------------------------------------
Ran 2 tests in 0.000s

FAILED (failures=1)


You may add `-v` onto the command line to have a more complete output, as an option of the `unittest` module: 

In [28]:
!cd code && {PY} -m unittest -v test_sum_unittest.py

test_sum (test_sum_unittest.TestSum) ... ok
test_sum_tuple (test_sum_unittest.TestSum) ... FAIL

FAIL: test_sum_tuple (test_sum_unittest.TestSum)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "d:\laveneau\OneDrive\OneDrive - Université de Poitiers\Enseignement\USTH\ProgrammingTechniques\Lectures\Lecture3\test_sum_unittest.py", line 8, in test_sum_tuple
    self.assertEqual(sum((1, 2, 2)), 6, "Should be 6")
AssertionError: 5 != 6 : Should be 6

----------------------------------------------------------------------
Ran 2 tests in 0.001s

FAILED (failures=1)


Well done: You have just executed two tests using the `unittest` test runner.

## Writing Your First Test
Let’s bring together what you’ve learned so far and, instead of testing the built-in `sum()` function, test a simple implementation of the same requirement.

For this purpose we would like to create a **Python module**:
- Create a new project folder called `my_sum`. 
- Inside my_sum, create an empty file called `__init__.py`. 

This is done by running the following cell:

In [11]:
%%script {PY}
import os
os.makedirs('code/my_sum', exist_ok=True)
open(os.path.join('code/my_sum', '__init__.py'), 'a').close()   # ensure the package marker exists

Creating the `__init__.py` file means that the `my_sum` folder can be imported as a module from the parent directory.
Then, we can add some functions and classes into this module, like it is done by running the next Python cell...

In [12]:
%%writefile code/my_sum/__init__.py
def sum(arg):
    total = 0
    for val in arg:
        total += val
    return total

Overwriting my_sum/__init__.py


This code example creates a variable called total, iterates over all the values in arg, and adds them to total. It then returns the result once the iterable has been exhausted.

### Where to Write the Test
To get started writing tests, you can simply create a file called `test.py`, which will contain your first test case. Because the file will need to be able to import your application before to test it, you want to place `test.py` above the package folder, for us in the current directory.

You’ll find that, as you add more and more tests, your single file will become cluttered and hard to maintain, so you can create a folder called `tests/` and split the tests into multiple files. 
It is convention to ensure each file starts with `test_` so all test runners will assume that Python file contains tests to be executed. 
Some very large projects split tests into more subdirectories based on their purpose or usage.

Run the next cell to build the directory...

In [13]:
%%script {PY}
import os
os.makedirs('code/tests', exist_ok=True)

### How to Structure a Simple Test
Before you dive into writing tests, you’ll want to first make a couple of decisions:
- What do you want to test?
- Are you writing a unit test or an integration test?

Then the structure of a test should loosely follow this workflow:
- Create your inputs.
- Execute the code being tested, capturing the output.
- Compare the output with an expected result.

For this application, you’re testing `sum()`. There are many behaviors in `sum()` you could check, such as:
- Can it sum a list of whole numbers (integers)?
- Can it sum a tuple or set?
- Can it sum a list of floats?
- What happens when you provide it with a bad value, such as a single integer or a string?
- What happens when one of the values is negative?

The most simple test would be a list of integers. This is done in the following cell:

In [31]:
%%writefile code/tests/test_sum.py
import unittest

from my_sum import sum

class TestSum(unittest.TestCase):
    def test_list_int(self):
        """
        Test that it can sum a list of integers
        """
        data = [1, 2, 3]
        result = sum(data)
        self.assertEqual(result, 6)

if __name__ == '__main__':
    unittest.main()

Overwriting ./tests/test_sum.py


### How to Write Assertions

The last step of writing a test is to validate the output against a known response. This is known as an assertion. There are some general best practices around how to **write assertions**:
- Make sure tests are repeatable and run your test multiple times to make sure it gives the same result every time.
- Try and assert results that relate to your input data, such as checking that the result is the actual sum of values in the `sum()` example.

`unittest` comes with lots of methods to assert on the values, types, and existence of variables. 
Here are some of the most commonly used methods:
| Method | Equivalent to |
| :- | :- |
| `.assertEqual(a, b)` | `a == b` |
| `.assertTrue(x)` | `bool(x) is True` |
| `.assertFalse(x)` | `bool(x) is False` |
| `.assertIs(a, b)` | `a is b` |
| `.assertIsNone(x)` | `x is None` |
| `.assertIn(a, b)` | `a in b` |
| `.assertIsInstance(a, b)` | `isinstance(a, b)` |

`.assertIs()`, `.assertIsNone()`, `.assertIn()`, and `.assertIsInstance()` all have opposite methods, named `.assertIsNot()`, and so forth.

### Side Effects
When you’re writing tests, it’s often not as simple as looking at the return value of a function. 
Often, executing a piece of code will alter other things in the environment, such as the attribute of a class, a file on the filesystem, or a value in a database. 
These are known as **side effects** and are an important part of testing. 
Decide if the side effect is being tested before including it in your list of assertions.

*Note: notice that this generally means a big problem in your application architecture, as the next lecture will try to explain it*. 

If you find that the unit of code you want to test has lots of side effects, you might be breaking the **Single Responsibility Principle**. Breaking the Single Responsibility Principle means the piece of code is doing too many things and would be better off being refactored. Following the Single Responsibility Principle is a great way to design code that it is easy to write repeatable and simple unit tests for, and ultimately, reliable applications.

## Running the tests    

### Executing Test Runners
Now, let us try to run this test. Actually, there are different ways to do that. 
Here we just present three of them:
1. By running it as Python script with `python tests/test_sum.py`.
2. By running it as a module with `python -m unittest tests/test_sum.py`.
3. By discovering and running all tests in a given directory with the following cell:

In [33]:
!cd code && {PY} -m unittest discover -s tests

.
----------------------------------------------------------------------
Ran 1 test in 0.000s

OK


### Understanding Test Output

That was a very simple example where everything passes, so now you’re going to try a failing test and interpret the output.

`sum()` should be able to accept other lists of numeric types, like fractions.

At the top of the `test_sum.py` file, we add an import statement to import the `Fraction` type from the `fractions` module in the standard library:
```python
from fractions import Fraction
```
Now we can add a test with an assertion expecting the **incorrect value**, in this case expecting the sum of `1/4`, `1/4`, and `2/5` to be 1 (the truth being `9/10`):

In [24]:
%%writefile code/tests/test_sum.py
import unittest

from fractions import Fraction
from my_sum import sum


class TestSum(unittest.TestCase):
    def test_list_int(self):
        """
        Test that it can sum a list of integers
        """
        data = [1, 2, 3]
        result = sum(data)
        self.assertEqual(result, 6)

    def test_list_fraction(self):
        """
        Test that it can sum a list of fractions
        """
        data = [Fraction(1, 4), Fraction(1, 4), Fraction(2, 5)]
        result = sum(data)
        self.assertEqual(result, 1)

if __name__ == '__main__':
    unittest.main()

Overwriting tests/test_sum.py


In [25]:
!cd code && {PY} -m unittest discover -s tests

F.
FAIL: test_list_fraction (test_sum.TestSum)
Test that it can sum a list of fractions
----------------------------------------------------------------------
Traceback (most recent call last):
  File "d:\laveneau\OneDrive\OneDrive - Université de Poitiers\Enseignement\USTH\ProgrammingTechniques\Lectures\Lecture3\tests\test_sum.py", line 22, in test_list_fraction
    self.assertEqual(result, 1)
AssertionError: Fraction(9, 10) != 1

----------------------------------------------------------------------
Ran 2 tests in 0.001s

FAILED (failures=1)


In the output, you can see the following information:
1. The first line shows the execution results of all the tests, `F.`: one failed (`F`) and one passed (`.`).
2. The `FAIL` entry shows some details about the failed test:
    - The test method name (`test_list_fraction`).
    - The test module (`test_sum`) and the test case (`TestSum`).
    - A `traceback` to the failing line.
    - The details of the assertion with the expected result (`1`) and the actual result (`Fraction(9, 10)`).

Again, we may add extra information to the test output by adding the `-v` flag to the `unittest` module as in the following cell.

In [26]:
!cd code && {PY} -m unittest discover -s tests -v

test_list_fraction (test_sum.TestSum)
Test that it can sum a list of fractions ... FAIL
test_list_int (test_sum.TestSum)
Test that it can sum a list of integers ... ok

FAIL: test_list_fraction (test_sum.TestSum)
Test that it can sum a list of fractions
----------------------------------------------------------------------
Traceback (most recent call last):
  File "d:\laveneau\OneDrive\OneDrive - Université de Poitiers\Enseignement\USTH\ProgrammingTechniques\Lectures\Lecture3\tests\test_sum.py", line 22, in test_list_fraction
    self.assertEqual(result, 1)
AssertionError: Fraction(9, 10) != 1

----------------------------------------------------------------------
Ran 2 tests in 0.001s

FAILED (failures=1)


## More Advanced Testing Scenarios
Before you step into creating tests for your application, remember the three basic steps of every test:
- Create your inputs.
- Execute the code, capturing the output.
- Compare the output with an expected result.

It is not always as easy as creating a static value for the input like a `string` or a `number`. 
Sometimes, your application will require an instance of a class or a context. 
What do you do then?

The data that you create as an input is known as **a fixture**. 
It’s common practice to create fixtures and reuse them.

If you are running the same test and passing different values each time and expecting the same result, this is known **as parameterization**.

### Handling Expected Failures
Earlier, when you made a list of scenarios to test `sum()`, a question came up: 
What happens when you provide it with a bad value, such as a single integer or a string?

In this case, you would expect `sum()` to throw an error. 
When it does throw an error, that would cause the test to fail.

There is a special way to *handle expected errors*. 
You can use `.assertRaises()` as a context-manager, then inside the with block execute the test steps:
```python
class TestSum(unittest.TestCase):
    ...
    def test_bad_type(self):
        data = "banana"
        with self.assertRaises(TypeError):
            result = sum(data)

```
This test case will now only pass if `sum(data)` raises a `TypeError`. 
You can replace `TypeError` with any exception type you choose.

### Isolating Behaviors in Your Application
Earlier in the lecture, you learned what a side effect is. 
Side effects make unit testing harder since, each time a test is run, it might give a different result, or even worse, one test could impact the state of the application and cause another test to fail!

There are some simple techniques you can use to test parts of your application that have many side effects:
- Refactoring code to follow the *Single Responsibility Principle* (see lecture 4).
- **Mocking** out any method or function calls to remove side effects.
- Using *integration testing* instead of *unit testing* for this piece of the application.

While Single Responsibility Principle is covered in the next lecture, and doing integration testing is not really a solution (because we still need to implement the unit tests!), let us take a look at the mocking solution coming with the `unittest` module.



### Using `unittest.mocking`
This section is based onto [https://docs.python.org/3/library/unittest.mock.html](https://docs.python.org/3/library/unittest.mock.html).

`unittest.mock` provides a core `Mock` class removing the need to create a host of stubs throughout your test suite. 
After performing an action, you can make assertions about which methods / attributes were used and arguments they were called with. 
You can also specify return values and set needed attributes in the normal way.

Additionally, `mock` provides a `patch()` *decorator* that handles patching module and class level attributes within the scope of a test, along with sentinel for creating unique objects. 

`Mock` is based on the ‘`action -> assertion`’ pattern instead of ‘`record -> replay`’ used by many mocking frameworks.

`Mock` and `MagicMock` objects create all attributes and methods as you access them and store details of how they have been used. You can configure them, to specify return values or limit what attributes are available, and then make assertions about how they have been used:
```python
from unittest.mock import MagicMock

thing = ProductionClass()
thing.method = MagicMock(return_value=3)
thing.method(3, 4, 5, key='value')
thing.method.assert_called_with(3, 4, 5, key='value')
```
This example will produce a single line output:
```bash
3
```
`side_effect` allows you to perform side effects, including raising an exception when a mock is called:
```python
mock = Mock(side_effect=KeyError('foo'))
mock()
```
It produces:
```bash
Traceback (most recent call last):
 ...
KeyError: 'foo'
```
Another `side_effect` example is at follows:
```python
def side_effect(arg):
    values = {'a': 1, 'b': 2, 'c': 3}
    return values[arg]

mock.side_effect = side_effect
mock('a'), mock('b'), mock('c')

mock.side_effect = [5, 4, 3, 2, 1]
mock(), mock(), mock()
```
It produces as output:
```bash
(1, 2, 3)
(5, 4, 3)
```
`Mock` has many other ways you can configure it and control its behaviour. 
For example the `spec` argument configures the mock to take its specification from another object. 
Attempting to access attributes or methods on the mock that don’t exist on the `spec` will fail with an `AttributeError`.

The `patch()` decorator / context manager makes it easy to mock classes or objects in a module under test. 
The object you specify will be replaced with a `mock` (or other object) during the test and restored when the test ends:
```python
from unittest.mock import patch

@patch('module.ClassName2')
@patch('module.ClassName1')
def test(MockClass1, MockClass2):
    module.ClassName1()
    module.ClassName2()
    assert MockClass1 is module.ClassName1
    assert MockClass2 is module.ClassName2
    assert MockClass1.called
    assert MockClass2.called

test()
```
Notice the patch decorator order: 
When you nest patch decorators the mocks are passed in to the decorated function in the same order they applied (the normal Python order that decorators are applied). This means from the bottom up, so in the example above the mock for `module.ClassName1` is passed in first.

With `patch()` it matters that you patch objects in the namespace where they are looked up. 
This is normally straightforward, but for a quick guide read where to patch.

As well as a decorator `patch()` can be used as a context manager in a with statement:
```python
with patch.object(ProductionClass, 'method', return_value=None) as mock_method:
    thing = ProductionClass()
    thing.method(1, 2, 3)

mock_method.assert_called_once_with(1, 2, 3)
```
There is also `patch.dict()` for setting values in a dictionary just during a scope and restoring the dictionary to its original state when the test ends:
```python
foo = {'key': 'value'}
original = foo.copy()
with patch.dict(foo, {'newkey': 'newvalue'}, clear=True):
    assert foo == {'newkey': 'newvalue'}

assert foo == original
```
Mock supports the mocking of Python magic methods. 
The easiest way of using magic methods is with the `MagicMock` class. 
It allows you to do things like:
```python
mock = MagicMock()
mock.__str__.return_value = 'foobarbaz'
str(mock)
mock.__str__.assert_called_with()
```
that produces a single output:
```bash
'foobarbaz'
```
Notice that the assertion is not raised, since the method was called just before...

Mock allows you to assign functions (or other Mock instances) to magic methods and they will be called appropriately. 
The MagicMock class is just a Mock variant that has all of the magic methods pre-created for you (well, all the useful ones anyway).

The following is an example of using magic methods with the ordinary Mock class:
```python
mock = Mock()
mock.__str__ = Mock(return_value='wheeeeee')
str(mock)
```
that produces as output:
```bash
'wheeeeee'
```

For ensuring that the `mock` objects in your tests have the same api as the objects they are replacing, you can use auto-speccing. 
Auto-speccing can be done through the `autospec` argument to `patch`, or the `create_autospec()` function. Auto-speccing creates mock objects that have the same attributes and methods as the objects they are replacing, and any functions and methods (including constructors) have the same call signature as the real object.

This ensures that your mocks will fail in the same way as your production code if they are used incorrectly:
```python
from unittest.mock import create_autospec

def function(a, b, c):
    pass

mock_function = create_autospec(function, return_value='fishy')
mock_function(1, 2, 3)
mock_function.assert_called_once_with(1, 2, 3)
mock_function('wrong arguments')
```
that will produces the following two outputs:
```bash
'fishy'
Traceback (most recent call last):
 ...
TypeError: <lambda>() takes exactly 3 arguments (1 given)
```
`create_autospec()` can also be used on classes, where it copies the signature of the `__init__` method, and on callable objects where it copies the signature of the `__call__` method.

## Testing with `pytest`

Everything above used `unittest`, and for a good reason: it ships with Python, so it is always there, and it is what this week's labworks and the automatic-differentiation project use.
But the Python world at large does not run on `unittest` — it runs on **`pytest`**, a third-party test framework you will meet in essentially every open-source project and in every company.
You will meet it in this course too: the optimisation lab of week 2 is a `pytest` suite.

The good news, and the reason we can afford to present it in half a page: **`pytest` is a test *runner* first, and a test *framework* second**.
It collects and runs `unittest.TestCase` classes unchanged.
Nothing you learned above is lost — you can point `pytest` at the very tests you have just written and they run.

Unlike `unittest`, `pytest` is not in the standard library, so it must be installed:
```bash
python -m pip install pytest
```

### A test is just a function, an assertion is just `assert`

In `pytest`, you do not subclass anything and you do not look up an assertion method.
A test is a **function** whose name starts with `test_`, living in a file whose name starts with `test_`, and it checks things with the plain Python `assert` statement:

In [ ]:
%%writefile code/tests/test_sum_pytest.py
from fractions import Fraction

from my_sum import sum


def test_list_int():
    """Test that it can sum a list of integers."""
    assert sum([1, 2, 3]) == 6


def test_list_fraction():
    """Test that it can sum a list of fractions -- correctly this time: 1/4 + 1/4 + 2/5 = 9/10."""
    assert sum([Fraction(1, 4), Fraction(1, 4), Fraction(2, 5)]) == Fraction(9, 10)


No `import unittest`, no `class`, no `self`, no `unittest.main()` at the bottom — and no table of `assertEqual` / `assertIn` / `assertIsInstance` to remember, because `assert a == b` and `assert a in b` and `assert isinstance(a, b)` are just Python.

Now run it. Note that we call it as `python -m pytest` and not simply `pytest`: exactly as with `python -m unittest`, the `-m` form puts the **current directory on `sys.path`**, which is what lets the test file import our `my_sum` package.

In [ ]:
!cd code && {PY} -m pytest tests -v

Read that output carefully, because `pytest` collected **four** tests and we only wrote two.
The other two are `TestSum.test_list_int` and `TestSum.test_list_fraction`, from the `tests/test_sum.py` written earlier in this lecture: a `unittest.TestCase`, collected and executed by `pytest` without a single change.
That is the compatibility promise in action — adopting `pytest` never means rewriting an existing `unittest` suite.

And `TestSum.test_list_fraction` still fails, exactly as it did under `python -m unittest discover`, since it is the test we deliberately wrote with the wrong expected value.
Its message is still the `unittest` one, `AssertionError: Fraction(9, 10) != 1`, because that text comes from `assertEqual` itself and not from the runner. The richer failure reports shown in the next section belong to the bare `assert` of `pytest`-style tests; what a mixed suite gains here is simply that both styles run under one command.

Try `-q` instead of `-v` for the short report (one character per test, as `unittest` does), and `python -m pytest` with no argument at all to discover every test under the current directory — `pytest` needs no equivalent of `discover -s tests`, since discovery is its default behaviour.

### Reading a failure

The real argument for `pytest` is not the shorter test, it is the **failure message**.
`self.assertEqual(a, b)` can tell you `a != b` because it was given both values.
A bare `assert` should not be able to tell you anything at all... yet `pytest` rewrites the assertions in your test files at import time so that it can take them apart and show you the intermediate values.
Let us provoke a failure on purpose, exactly as we did with `unittest` at the beginning of the lecture:

In [ ]:
%%writefile code/test_sum_fail_pytest.py
from my_sum import sum


def test_sum_tuple():
    # intentionally wrong: (1, 2, 2) sums to 5, so this test FAILS on purpose
    data = (1, 2, 2)
    assert sum(data) == 6, "Should be 6"


def test_a_harder_comparison():
    # also wrong on purpose: look at how the two lists are compared below
    assert [1, 2, 3, 4] == [1, 2, 4, 3]


In [ ]:
!cd code && {PY} -m pytest test_sum_fail_pytest.py

For the first failure you get the offending line, the value of `data`, and the value of `sum(data)`.
For the second you get a *diff* telling you which index of the list differs.
This introspection is free and works on any expression you write — comparisons, `in`, `isinstance`, attribute access, a function call.

Two things are worth knowing about failures:
- `-x` stops at the first failure instead of running the whole suite;
- `--tb=short` (or `--tb=line`, or `--tb=no`) shrinks the traceback when the output gets long.

### Expected exceptions and floating-point values

The `unittest` context manager `self.assertRaises(TypeError)` becomes `pytest.raises(TypeError)`, with the bonus that it hands you the exception object so you can assert on its message:
```python
import pytest

def test_bad_type():
    with pytest.raises(TypeError):
        sum("banana")

def test_error_message():
    with pytest.raises(ValueError, match="must be positive"):
        area(-1.0)
```

Floating-point results must **never** be compared with `==`, since `0.1 + 0.2 != 0.3` in binary floating point.
`unittest` offers `assertAlmostEqual`; `pytest` offers `pytest.approx`, which also works element-wise on lists, tuples, dictionaries and `numpy` arrays:
```python
assert 0.1 + 0.2 == pytest.approx(0.3)
assert [0.1 + 0.2, 1.0 / 3] == pytest.approx([0.3, 0.333333333])
assert loss.value(w) == pytest.approx(5.0, rel=1e-9)
```
You will see `pytest.approx` on nearly every line of the week-2 optimisation tests, for the obvious reason that the whole lab computes with floats.

### Fixtures

Remember the definition given above: *a fixture is the data you create as an input*, and it is common practice to create fixtures and reuse them.
`unittest` gives you `setUp()`, which runs before every test method of a `TestCase` and stores its results on `self`.

`pytest` turns the fixture into a **named function**, and a test asks for the fixtures it needs simply by **naming them as parameters**:
```python
import pytest

@pytest.fixture
def data():
    """The three-element list used by several tests below."""
    return [1, 2, 3]

def test_sum(data):          # <- pytest sees the name "data" and calls the fixture
    assert sum(data) == 6

def test_length(data):       # <- a *fresh* list, rebuilt for this test
    assert len(data) == 3
```
This is more than cosmetic. A test declares exactly the inputs it depends on, instead of inheriting every attribute that `setUp()` happened to create; two tests that need different inputs no longer have to live in different classes; and a fixture may itself request other fixtures, so expensive objects can be built in layers.

A fixture placed in a file named **`conftest.py`** is visible to every test file in that directory and below, with no import.
That is how the week-2 lab shares its small hand-computed problem across the whole suite — go and read `Week2/optlab/tests/conftest.py` once you get there:
```python
# conftest.py
@pytest.fixture
def rng() -> np.random.Generator:
    """Seeded, so a failure is reproducible."""
    return np.random.default_rng(20250921)
```

Fixtures can also clean up after themselves: write `yield` instead of `return`, and whatever follows the `yield` runs after the test (this is `tearDown()`).
And the `scope=` argument (`"function"` by default, or `"module"`, `"session"`) says how often the fixture is rebuilt.

### Parameterization

Also defined above: *running the same test and passing different values each time* is parameterization.
With `unittest` this means a loop inside the test (and then one failure hides the others) or the `subTest` context manager.
`pytest` has a decorator, and each case is reported as a **separate test**:
```python
@pytest.mark.parametrize("data, expected", [
    ([1, 2, 3], 6),
    ((1, 2, 3), 6),
    ([0.1, 0.2], 0.3),
    ([], 0),
])
def test_sum(data, expected):
    assert sum(data) == pytest.approx(expected)
```
Four tests, one function body. When the third one fails you are told `test_sum[data2-0.3]` failed, and the other three still ran.

### Markers, or selecting a subset of the suite

`@pytest.mark.<name>` attaches a label to a test, and `-m` runs only the tests carrying a given label:
```python
@pytest.mark.slow
def test_convergence_on_a_big_problem():
    ...
```
```bash
python -m pytest -m "not slow"          # skip the slow ones
python -m pytest -m "day1 or day2"      # only the first two days
```
`pytestmark = pytest.mark.day1` at the top of a file marks **every** test in that file at once.
Markers must be declared in `pyproject.toml` (under `[tool.pytest.ini_options] markers = [...]`), otherwise `pytest` warns about a typo'd name — which is exactly the point.

This is not an exotic feature: it is how the week-2 lab lets you run the tests of the material you have actually covered, `make test DAY=3` expanding to `pytest -m "day1 or day2 or day3"`, so that the exercises of day 5 do not report as failures on day 3.

Two markers are built in and worth knowing now:
- `@pytest.mark.skip(reason="...")` and `@pytest.mark.skipif(condition, reason="...")` — do not run this test;
- `@pytest.mark.xfail(reason="...")` — this test is *expected to fail*; it is reported as `x` when it fails and, more usefully, as `XPASS` when it unexpectedly passes. This is the `unittest` `@expectedFailure` of the section above.

### The translation table

| `unittest` | `pytest` |
| :- | :- |
| `class TestX(unittest.TestCase)` | a module of `test_*` functions (no class needed) |
| `self.assertEqual(a, b)` | `assert a == b` |
| `self.assertTrue(x)` / `self.assertFalse(x)` | `assert x` / `assert not x` |
| `self.assertIs(a, b)` / `self.assertIsNone(x)` | `assert a is b` / `assert x is None` |
| `self.assertIn(a, b)` / `self.assertIsInstance(a, b)` | `assert a in b` / `assert isinstance(a, b)` |
| `self.assertAlmostEqual(a, b)` | `assert a == pytest.approx(b)` |
| `with self.assertRaises(E):` | `with pytest.raises(E):` |
| `setUp()` / `tearDown()` | `@pytest.fixture` (with `yield` for the teardown) |
| `@unittest.expectedFailure` | `@pytest.mark.xfail` |
| `@unittest.skipIf(cond, reason)` | `@pytest.mark.skipif(cond, reason=...)` |
| `with self.subTest(...)` in a loop | `@pytest.mark.parametrize` |
| `python -m unittest discover -s tests` | `python -m pytest` |

Note the one row that is **missing**: mocking.
There is no `pytest` mocking library to learn, because `unittest.mock` — `Mock`, `MagicMock`, `patch`, `side_effect`, `create_autospec`, everything in the section above — is part of the standard library and is used as-is from `pytest` tests.
The mocking you have just learned is the mocking you will use, whichever runner you pick.

### Which one do we use in this course?

Both, deliberately, and it is always written down:

| Where | Framework | How it is run |
| :- | :- | :- |
| Week 1 — labwork 3 | `unittest` | `python -m unittest discover -s tests` |
| Week 1 — the `autodiff` project (labwork 5) | `unittest` | `make test` |
| Week 2 — the `optlab` optimisation lab | `pytest` | `make test DAY=n` |

Week 1 stays on `unittest` because it needs nothing installed and because `TestCase` makes the vocabulary — test case, fixture, assertion, isolation — explicit before the syntax gets short.
Week 2 moves to `pytest` because its suite genuinely needs what `pytest` adds: `approx` on every float comparison, fixtures shared through `conftest.py`, and day markers to run only the material already covered.

If you want to convince yourself that the two are not in conflict, go back to your labwork 3 or to the `autodiff` project, install `pytest`, and run `python -m pytest` in the project directory.
Your `unittest` tests will run, unchanged, and you will get the nicer failure messages for free.

## Conclusions
This lecture just gave you a short overview of the testing problem and the two Python tools used to accomplish it, `unittest` and `pytest`.
It has not covered integration testing, data-driven testing, continuous integration (e.g. GitHub Actions or GitLab CI), in-depth mocking, code coverage, full functional testing, testing across operating systems, and so on.

That's a good news: you still have so many things to discover &#x1F609;

Remember for the rest of the course: **week 1 -- this lecture's labwork and the `autodiff` project -- runs on `unittest`**, since it needs nothing installed; **week 2's optimisation lab runs on `pytest`**, since it needs `approx`, shared fixtures and day markers.
The mocking is `unittest.mock` in both cases.